# 05 — Manuscript figures

Generate the PPC defective-density figure and the combined v / a / DDM panel figure from de-identified trial data and group-level posterior InferenceData.

## Prerequisites

Prefer `hddm_analysis/notebooks/` (package root via `_resolve_hddm_root()`). Required inputs (auto-loaded from disk or in-memory if upstream notebooks were run):

| Input | Source |
|-------|--------|
| Behavioral trials | `../data/trials_hddm_ready.tsv` |
| Winning-model posterior | `../models/transition_prob_duration_10000samples/hddm_va_vat_combined.nc` |
| HDDM | Required for PPC posterior predictive checks (`hddm.generate.gen_rts`) |

## Outputs

Saved under `../figures/` as PNG, PDF, EPS, and SVG for each figure stem:

| Manuscript | Function | Output stem |
|------------|----------|-------------|
| PPC | `plot_figure_ppc` | `Figure_PPC_DefectiveDensity_2x4` |
| Combined panel | `plot_combined_figure_v_a_ddm` | `Figure_combined_v_a_DDM_2x3` |



In [1]:
import os
from pathlib import Path
import sys

import arviz as az
import pandas as pd

def _resolve_hddm_root() -> Path:
    """Locate hddm_analysis/ regardless of notebook cwd."""
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data" / "trials_hddm_ready.tsv").is_file():
            return candidate
        nested = candidate / "hddm_analysis"
        if (nested / "data" / "trials_hddm_ready.tsv").is_file():
            return nested
    if start.name == "notebooks" and (start.parent / "data").is_dir():
        return start.parent
    raise FileNotFoundError(
        "Could not locate hddm_analysis package root "
        "(expected data/trials_hddm_ready.tsv). "
        f"Started search from: {start}"
    )


ROOT = str(_resolve_hddm_root())
SCRIPTS_DIR = os.path.join(ROOT, "scripts")
if SCRIPTS_DIR not in sys.path:
    sys.path.insert(0, SCRIPTS_DIR)
from nc_io import from_netcdf as load_idata
DATA_DIR = os.path.join(ROOT, "data")
RESULTS_DIR = os.path.join(ROOT, "results")
MODEL_DIR = os.path.join(ROOT, "models", "transition_prob_duration_10000samples")
FIG_DIR = os.path.join(ROOT, "figures")
for _d in (DATA_DIR, RESULTS_DIR, MODEL_DIR, FIG_DIR):
    os.makedirs(_d, exist_ok=True)

# --- Figure configuration ---
WINNER_MODEL = "hddm_va_vat"
REQUIRE_HDDM_FOR_PPC = True
# Optional explicit override, e.g. r"D:\path\to\hddm_va_vat_combined.nc"
WINNER_IDATA_PATH = os.environ.get("HDDM_WINNER_IDATA_PATH", "").strip() or None


def flip_errors(df):
    """Flip RT sign for incorrect responses (HDDM convention)."""
    out = df.copy()
    out.loc[out["response"] != 1, "rt"] = -out.loc[out["response"] != 1, "rt"]
    return out


def save_figure(fig, savepath, **kwargs):
    """Save figure to PNG, PDF, EPS, and SVG."""
    stem, _ = os.path.splitext(savepath)
    base_kw = dict(kwargs)
    dpi = base_kw.pop("dpi", 300)
    base_kw.pop("transparent", None)
    for ext in ("png", "pdf", "eps", "svg"):
        path = f"{stem}.{ext}"
        if ext == "png":
            fig.savefig(path, transparent=True, dpi=dpi, **base_kw)
        elif ext == "eps":
            # PostScript backend does not support transparency.
            fig.savefig(path, format="eps", **base_kw)
        else:
            fig.savefig(path, transparent=True, **base_kw)
        print(f"Saved: {path}")


def resolve_hddm_data():
    """Load HDDM-ready behavioral data from memory or bundled TSV."""
    if "hddm_data" in globals() and isinstance(globals()["hddm_data"], pd.DataFrame):
        df = globals()["hddm_data"]
        required = {"rt", "response", "long_short", "prob", "transition"}
        if required.issubset(df.columns):
            return df.copy()
    tsv = os.path.join(DATA_DIR, "trials_hddm_ready.tsv")
    if not os.path.isfile(tsv):
        raise RuntimeError(
            f"Behavioral data not found. Place bundled trials at:\n  {tsv}"
        )
    trials = pd.read_csv(tsv, sep="\t")
    trials = trials.loc[trials["rt_ms"] > 150].copy()
    df = trials.copy()
    df["response"] = df["acc"].astype(int)
    df["rt"] = df["rt_ms"] / 1000.0
    df = df[["subj_idx", "transition", "prob", "long_short", "response", "rt"]]
    return flip_errors(df)


def _candidate_idata_paths():
    """Search order for winning-model InferenceData on disk."""
    fname = f"{WINNER_MODEL}_combined.nc"
    if WINNER_IDATA_PATH:
        yield WINNER_IDATA_PATH
    for d in (MODEL_DIR,):
        if d:
            yield os.path.join(d, fname)


def resolve_best_idata():
    """Load winning-model InferenceData from memory or local NetCDF."""
    if "best_idata" in globals() and isinstance(globals()["best_idata"], az.InferenceData):
        return globals()["best_idata"]

    searched = []
    for nc_path in _candidate_idata_paths():
        searched.append(nc_path)
        if os.path.isfile(nc_path):
            print(f"Loading InferenceData: {nc_path}")
            return load_idata(nc_path)

    msg = "\n".join(f"  - {p}" for p in searched)
    raise RuntimeError(
        f"Winning-model InferenceData not found. Searched:\n{msg}\n"
        ""
        "Set WINNER_IDATA_PATH or env HDDM_WINNER_IDATA_PATH to override."
    )


In [2]:
# ==============================
# PPC figure + combined-panel definitions
# Winning model: hddm_va_vat (v,a depend on condition; t free scalar)
# ==============================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import arviz as az
import logging
from pathlib import Path

from matplotlib.lines import Line2D
from matplotlib import font_manager as mfont
from matplotlib import gridspec
from scipy.stats import gaussian_kde

# ---------- Combined-panel styling: Arial, spine bounds, offset axes ----------
_FIG235_ARIAL_REGISTERED = False


def _register_arial_fonts():
    """Register Arial.ttf from standard paths so the 'Arial' family resolves when present."""
    global _FIG235_ARIAL_REGISTERED
    if _FIG235_ARIAL_REGISTERED:
        return
    candidates = []
    if os.name == "nt":
        windir = os.environ.get("WINDIR", r"C:\Windows")
        candidates.extend(
            [
                os.path.join(windir, "Fonts", "arial.ttf"),
                os.path.join(windir, "Fonts", "ARIAL.TTF"),
            ]
        )
    candidates.extend(
        [
            "/usr/share/fonts/truetype/msttcorefonts/Arial.ttf",
            "/usr/share/fonts/truetype/microsoft/Arial.ttf",
            "/Library/Fonts/Microsoft/Arial.ttf",
            "/Library/Fonts/Arial.ttf",
            "/System/Library/Fonts/Supplemental/Arial.ttf",
        ]
    )
    for path in candidates:
        if path and os.path.isfile(path):
            try:
                mfont.fontManager.addfont(path)
            except Exception:
                continue
    _FIG235_ARIAL_REGISTERED = True


def _clip_spines_to_tick_range(ax):
    """Clip bottom/left spines to the min/max *visible* major ticks (no line past outer ticks)."""
    xlo, xhi = ax.get_xlim()
    xt = np.asarray(ax.get_xticks(), dtype=float)
    xt = xt[np.isfinite(xt) & (xt >= xlo - 1e-12) & (xt <= xhi + 1e-12)]
    if xt.size:
        ax.spines["bottom"].set_bounds(float(np.min(xt)), float(np.max(xt)))
    ylo, yhi = ax.get_ylim()
    yt = np.asarray(ax.get_yticks(), dtype=float)
    yt = yt[np.isfinite(yt) & (yt >= ylo - 1e-12) & (yt <= yhi + 1e-12)]
    if yt.size:
        ax.spines["left"].set_bounds(float(np.min(yt)), float(np.max(yt)))


def _style_spines_offset(ax, outward=4):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    lw = float(mpl.rcParams.get("axes.linewidth", 0.9))
    for side in ("left", "bottom"):
        ax.spines[side].set_visible(True)
        ax.spines[side].set_linewidth(lw)
        ax.spines[side].set_position(("outward", outward))


def _fig235_rc_context(font_pt_bump: float = 0):
    from matplotlib.font_manager import FontProperties, findfont

    _register_arial_fonts()
    path_try = findfont(FontProperties(family="Arial"))
    use_arial = bool(path_try) and ("arial" in os.path.basename(path_try).lower())
    if use_arial:
        math_rc = {
            "mathtext.fontset": "custom",
            "mathtext.rm": "Arial",
            "mathtext.it": "Arial:italic",
            "mathtext.bf": "Arial:bold",
        }
    else:
        math_rc = {"mathtext.fontset": "dejavusans"}
    b = float(font_pt_bump)
    return mpl.rc_context(
        {
            "font.family": "sans-serif",
            "font.sans-serif": ["Arial", "Liberation Sans", "Nimbus Sans", "Helvetica", "DejaVu Sans"],
            "font.size": 9 + b,
            "axes.titlesize": 10 + b,
            "axes.labelsize": 9 + b,
            "xtick.labelsize": 8 + b,
            "ytick.labelsize": 8 + b,
            "legend.fontsize": 8 + b,
            **math_rc,
        }
    )


_register_arial_fonts()
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)


# ---------- matplotlib style ----------
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Liberation Sans", "Nimbus Sans", "Helvetica", "DejaVu Sans"],
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "axes.linewidth": 0.9,
    "xtick.major.width": 0.9,
    "ytick.major.width": 0.9,
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.transparent": True,
    "savefig.facecolor": "none",
})

RNG = np.random.default_rng(20260425)

# ---------- experimental condition labels ----------
DURATION_LOW_DENSITY, DURATION_HIGH_DENSITY = "Long", "Short"          # (Predictability/Density axis)
PROB_LOW_PERT, PROB_HIGH_PERT = "prob_same", "prob_diff"         # (Perturbation axis)
TRANS_FLEX, TRANS_STAB = "flexibility", "stability"              # (Control domain)

# Combined-figure palette
C_FLEX = "#E6C4B5"           # Flexibility (panels A, D) — muted peach (raincloud Task)
C_STAB = "#9CAABF"           # Stability (panels A, D) — muted blue-grey (raincloud Distractor)
C_LOW_DENSITY = "#85A897"       # Low Density (B,C,D) — DDM panel E left border (green)
C_HIGH_DENSITY = "#9B8AA6"      # High Density (B,C,D) — DDM panel E right border (purple)

FIG_OUTDIR = FIG_DIR
os.makedirs(FIG_OUTDIR, exist_ok=True)


def _cell(d, p, t):
    return f"{d}.{p}.{t}"


def _kde(x, n=400, bw_factor=1.0):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if x.size < 5:
        return np.array([0.0, 1.0]), np.array([0.0, 0.0])
    kde = gaussian_kde(x)
    if bw_factor != 1.0:
        kde.set_bandwidth(kde.factor * float(bw_factor))
    lo, hi = float(np.min(x)), float(np.max(x))
    pad = (hi - lo) * 0.15 + 1e-3
    xs = np.linspace(lo - pad, hi + pad, n)
    ys = kde(xs)
    return xs, ys


def _hdi(x, p=0.95):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if x.size < 2:
        return (np.nan, np.nan)
    h = np.asarray(az.hdi(x, hdi_prob=p), dtype=float).ravel()
    return float(h[0]), float(h[1])


def _posterior_flat(idata, varname, max_flat=20000):
    a = np.asarray(idata.posterior[varname].values)
    return a.reshape(-1)[:max_flat]


def _get_var(idata, varname):
    # exact match only (keeps robust + predictable)
    if varname in idata.posterior:
        return _posterior_flat(idata, varname)
    return None


def _traces_8(idata, param):
    out = {}
    for d in (DURATION_LOW_DENSITY, DURATION_HIGH_DENSITY):
        for p in (PROB_LOW_PERT, PROB_HIGH_PERT):
            for t in (TRANS_FLEX, TRANS_STAB):
                c = _cell(d, p, t)
                v = _get_var(idata, f"{param}({c})")
                if v is not None:
                    out[c] = v
    return out


def _main_marginal(traces_by_cell, factor_idx, level, sep="."):
    arrs = [v for k, v in traces_by_cell.items() if k.split(sep)[factor_idx] == level]
    return np.mean(np.stack(arrs, axis=0), axis=0)



if REQUIRE_HDDM_FOR_PPC:
    try:
        from hddm.generate import gen_rts as _hddm_gen_rts
    except ImportError as _hddm_err:
        raise ImportError(
            "hddm is required for PPC posterior predictive checks. "
            "Install HDDM (see requirements.txt)."
        ) from _hddm_err
else:
    _hddm_gen_rts = None


# PPC posterior draws per cell (override with env HDDM_PPC_N). 1500 is enough for
# smooth defective-density envelopes; 6000*8 sequential gen_rts can take many minutes.
PPC_N = int(os.environ.get('HDDM_PPC_N', '1500'))


def _ppc_sim_from_posterior(idata, cell, n=None):
    """Posterior predictive RT samples for mirrored PPC (via hddm.generate.gen_rts).

    Returns rt_cor and rt_err (seconds, positive magnitudes).
    """
    if n is None:
        n = PPC_N
    v = _get_var(idata, f"v({cell})")
    a = _get_var(idata, f"a({cell})")
    t = _get_var(idata, "t")
    if v is None or a is None:
        return np.array([]), np.array([])
    if t is None:
        t = np.full_like(v, 0.3)

    m = int(min(n, len(v), len(a), len(t)))
    idx = RNG.choice(int(min(len(v), len(a), len(t))), size=m, replace=False)

    if _hddm_gen_rts is None:
        raise RuntimeError(
            "HDDM is required for posterior predictive simulation (PPC figure)."
        )

    rt_cor = []
    rt_err = []
    for ii in idx:
        d = _hddm_gen_rts(
            size=1,
            method="cdf",
            v=float(v[ii]),
            a=float(a[ii]),
            t=float(t[ii]),
            sv=0,
            sz=0,
            st=0,
        )
        rt = float(d["rt"].iloc[0])
        resp = int(d["response"].iloc[0])
        if resp == 1:
            rt_cor.append(rt)
        else:
            rt_err.append(rt)
    return np.asarray(rt_cor, float), np.asarray(rt_err, float)


def _smooth_defective_curve(samples_ms, edges, p_mass, sigma_bins=1.2):
    """Return (centers, smoothed defective-density y) where integral ≈ p_mass.

    Works even when sample size is tiny; if no samples, returns zeros.
    """
    samples_ms = np.asarray(samples_ms, float)
    samples_ms = samples_ms[np.isfinite(samples_ms)]
    centers = 0.5 * (edges[1:] + edges[:-1])
    if samples_ms.size == 0 or p_mass <= 0:
        return centers, np.zeros_like(centers)

    hist, _ = np.histogram(samples_ms, bins=edges, density=True)

    # Gaussian smoothing in "bin space" (no SciPy dependency)
    rad = int(np.ceil(4 * sigma_bins))
    x = np.arange(-rad, rad + 1)
    ker = np.exp(-0.5 * (x / sigma_bins) ** 2)
    ker = ker / np.sum(ker)
    smooth = np.convolve(hist, ker, mode="same")

    # convert to defective density by multiplying choice mass
    y = smooth * float(p_mass)
    return centers, y


def _plot_defective_mirror(ax, rt_signed, rt_cor_ppc, rt_err_ppc, x_max_ms=2000, bins=35):
    """Mirrored defective density: upper=correct, lower=error; areas sum to 1.

    - Observed: light-gray mirrored histogram (defective density)
    - PPC: dark smoothed envelope for BOTH correct and error (always drawn)
    """
    rt_signed = np.asarray(rt_signed, float)
    rt_signed = rt_signed[np.isfinite(rt_signed)]

    cor = rt_signed[rt_signed > 0] * 1000.0
    err = (-rt_signed[rt_signed < 0]) * 1000.0
    n = max(1, cor.size + err.size)
    p_cor = cor.size / n
    p_err = err.size / n

    edges = np.linspace(0.0, float(x_max_ms), bins + 1)
    centers = 0.5 * (edges[1:] + edges[:-1])
    width = edges[1] - edges[0]

    # observed (hist): light gray
    if cor.size:
        h, _ = np.histogram(cor, bins=edges, density=True)
        ax.bar(
            centers,
            h * p_cor,
            width=width,
            color="#bdbdbd",
            alpha=0.55,
            edgecolor="none",
            align="center",
            zorder=1,
        )
    if err.size:
        h, _ = np.histogram(err, bins=edges, density=True)
        ax.bar(
            centers,
            -(h * p_err),
            width=width,
            color="#bdbdbd",
            alpha=0.55,
            edgecolor="none",
            align="center",
            zorder=1,
        )

    # PPC (envelope): always draw both halves
    cor_ppc = np.asarray(rt_cor_ppc, float)
    err_ppc = np.asarray(rt_err_ppc, float)
    cor_ppc = cor_ppc[np.isfinite(cor_ppc)] * 1000.0
    err_ppc = err_ppc[np.isfinite(err_ppc)] * 1000.0
    nppc = max(1, cor_ppc.size + err_ppc.size)
    p_cor_ppc = cor_ppc.size / nppc
    p_err_ppc = err_ppc.size / nppc

    xs, y_cor = _smooth_defective_curve(cor_ppc, edges, p_cor_ppc, sigma_bins=1.25)
    _, y_err = _smooth_defective_curve(err_ppc, edges, p_err_ppc, sigma_bins=1.25)
    ax.plot(xs, y_cor, color="#2c3e50", lw=1.4, zorder=3)
    ax.plot(xs, -y_err, color="#2c3e50", lw=1.4, zorder=3)

    ax.axhline(0, color="black", lw=0.7, alpha=0.6)
    ax.set_xlim(0, x_max_ms)


def plot_figure_ppc(idata, df, savepath=None):
    """PPC figure: 2×4 mirrored PPC for all 8 conditions (defective density).

    Note. Positive y = correct RT; negative y = error RT. Curves show posterior predictive distributions.
    """

    # Ensure RT is flipped convention (errors negative)
    df2 = df.copy()
    if df2["rt"].min() >= 0 and "response" in df2.columns:
        err = df2["response"].astype(int).to_numpy() == 0
        df2.loc[err, "rt"] *= -1

    df2["cell"] = df2.apply(
        lambda r: f"{r['long_short']}.{r['prob']}.{r['transition']}", axis=1
    )

    # x-range in ms (cap 0–2000ms as requested)
    x_max_ms = float(
        np.nanpercentile(
            np.abs(df2["rt"].to_numpy(float)) * 1000.0,
            99.5,
        )
    )
    x_max_ms = float(np.clip(x_max_ms, 800.0, 2000.0))

    fig = plt.figure(figsize=(12.4, 5.6))
    # spacing: reduce vertical to 2/3; horizontal to 1/3
    gs = gridspec.GridSpec(2, 4, hspace=0.275, wspace=0.117)

    # Map condition columns to paper-friendly labels
    perturb_label = {PROB_LOW_PERT: "Low Density", PROB_HIGH_PERT: "High Density"}
    predict_label = {DURATION_LOW_DENSITY: "Low Pred.", DURATION_HIGH_DENSITY: "High Pred."}
    control_label = {TRANS_FLEX: "Flexibility", TRANS_STAB: "Stability"}

    # cols: (LowPert-Flex, LowPert-Stab, HighPert-Flex, HighPert-Stab)
    col_specs = [
        (PROB_LOW_PERT, TRANS_FLEX),
        (PROB_LOW_PERT, TRANS_STAB),
        (PROB_HIGH_PERT, TRANS_FLEX),
        (PROB_HIGH_PERT, TRANS_STAB),
    ]
    # rows: Predictability Low/High (Long/Short)
    row_specs = [DURATION_LOW_DENSITY, DURATION_HIGH_DENSITY]

    for r, dur in enumerate(row_specs):
        for c, (prob, tran) in enumerate(col_specs):
            ax = fig.add_subplot(gs[r, c])
            cell = _cell(dur, prob, tran)
            obs = df2.loc[df2["cell"] == cell, "rt"].to_numpy(float)
            rt_cor_ppc, rt_err_ppc = _ppc_sim_from_posterior(idata, cell, n=PPC_N)
            _plot_defective_mirror(
                ax,
                obs,
                rt_cor_ppc,
                rt_err_ppc,
                x_max_ms=x_max_ms,
                bins=40,
            )

            # Title (1 line; levels only) — centered
            ttl = f"{perturb_label[prob]}, {predict_label[dur]}, {control_label[tran]}"
            # Title placement: use y-position (more reliable than pad under tight_layout)
            _t = ax.set_title(ttl, fontsize=9.5)
            _t.set_ha("center")
            _t.set_position((0.5, 1.10))  # raise title above axes

            # ticks
            ax.tick_params(labelsize=9)

            # y-axis label: only left column, use "Defective density"
            if c == 0:
                ax.set_ylabel("Defective Density\n(+ correct / − error)", fontsize=10)
            else:
                ax.set_yticklabels([])

            # x-axis label: only bottom row
            if r == 1:
                ax.set_xlabel("RT (ms)", fontsize=10)
            else:
                ax.set_xticklabels([])

    # legend + caption
    h = [
        Line2D([0], [0], color="#bdbdbd", lw=10, alpha=0.55),
        Line2D([0], [0], color="#2c3e50", lw=1.8),
    ]
    fig.legend(
        h,
        ["Observed", "Posterior predictive"],
        frameon=False,
        loc="lower center",
        ncol=2,
        fontsize=9,
        # move legend up by ~1/3 step
        bbox_to_anchor=(0.5, -0.02),
    )

    # (No big title, no caption)
    # leave a bit more headroom so raised titles don't get clipped
    fig.tight_layout(rect=(0, 0.08, 1, 0.95))
    if savepath:
        save_figure(
            fig, savepath, dpi=300, transparent=True, facecolor="none", edgecolor="none"
        )
    return fig


def _plot_combined_v_panels(idata, axes):
    """Combined figure row 1 (panels A–C): drift-rate main effects and contrasts."""
    axA, axB, axC = axes
    fig = axA.figure
    tv = _traces_8(idata, "v")
    if len(tv) < 8:
        raise RuntimeError("Missing some v(condition) nodes. Expected 8 design cells.")
        
    # Data extraction for Panel A
    flex = _main_marginal(tv, factor_idx=2, level=TRANS_FLEX)
    stab = _main_marginal(tv, factor_idx=2, level=TRANS_STAB)
    
    # Data extraction for Panel B
    def marg(d, p):
        return 0.5 * (tv[_cell(d,p,TRANS_FLEX)] + tv[_cell(d,p,TRANS_STAB)])
    s11 = marg(DURATION_LOW_DENSITY, PROB_LOW_PERT)
    s12 = marg(DURATION_LOW_DENSITY, PROB_HIGH_PERT)
    s21 = marg(DURATION_HIGH_DENSITY, PROB_LOW_PERT)
    s22 = marg(DURATION_HIGH_DENSITY, PROB_HIGH_PERT)

    # Data extraction for Panel C
    def cell(d, p, t):
        return tv[_cell(d, p, t)]
    delta_long = (cell(DURATION_LOW_DENSITY, PROB_LOW_PERT, TRANS_FLEX) - cell(DURATION_LOW_DENSITY, PROB_LOW_PERT, TRANS_STAB)) - (
        cell(DURATION_LOW_DENSITY, PROB_HIGH_PERT, TRANS_FLEX) - cell(DURATION_LOW_DENSITY, PROB_HIGH_PERT, TRANS_STAB)
    )
    delta_short = (cell(DURATION_HIGH_DENSITY, PROB_LOW_PERT, TRANS_FLEX) - cell(DURATION_HIGH_DENSITY, PROB_LOW_PERT, TRANS_STAB)) - (
        cell(DURATION_HIGH_DENSITY, PROB_HIGH_PERT, TRANS_FLEX) - cell(DURATION_HIGH_DENSITY, PROB_HIGH_PERT, TRANS_STAB)
    )

    with _fig235_rc_context(font_pt_bump=1):
        # ==========================================
        # Panel A: Optimized Raincloud (Half-Violin)
        # ==========================================
        data_A = [flex, stab]
        # Flexibility / Stability colors
        colors_A = [C_FLEX, C_STAB]
        labels_A = ["Flexibility", "Stability"]
    
        # Vertical raincloud: x = condition, y = drift rate v (right-half violin)
        vpt = axA.violinplot(
            data_A,
            positions=[0, 1],
            widths=0.92,
            showmeans=False,
            showextrema=False,
            showmedians=False,
            vert=True,
        )

        for i, (arr, col) in enumerate(zip(data_A, colors_A)):
            b = vpt["bodies"][i]
            b.set_facecolor(col)
            b.set_alpha(0.5)
            verts = b.get_paths()[0].vertices
            verts[:, 0] = np.maximum(verts[:, 0], float(i))

            n_points = min(250, arr.size)
            idx = RNG.choice(arr.size, size=n_points, replace=False)
            jitter = RNG.uniform(low=-0.2, high=-0.05, size=n_points)
            axA.scatter(i + jitter, arr[idx], s=4.0, alpha=0.3, color=col, edgecolor="none", zorder=1)

            median = np.median(arr)
            lo, hi = _hdi(arr, 0.95)
            axA.plot([i, i], [lo, hi], color="black", lw=1.5, zorder=2)
            axA.scatter(i, median, color="white", edgecolor="black", s=30, zorder=3)

        axA.set_xticks([0, 1])
        axA.set_xticklabels(labels_A, fontsize=10)
        axA.set_ylabel("Drift Rate ($\\mathit{v}$)", fontsize=12)
        axA.set_yticks(np.arange(1.5, 4.7 + 1e-9, 0.8))
        axA.set_ylim(1.5, 4.7)
        axA.yaxis.set_major_formatter(mpl.ticker.FormatStrFormatter("%.1f"))
        axA.tick_params(labelsize=10)

        # Panel label (will be aligned after tight_layout)
        _panelA = axA.text(0.0, 1.0, "a", transform=axA.transAxes, ha="left", va="center", fontweight="bold", fontsize=_PANEL_LETTER_FS)


        # ==========================================
        # Panel B: Predictability × Perturbation
        # ==========================================

        # Panel B: posterior trace spaghetti plot
        offs = (-0.1, 0.1)
        colors_B = (C_LOW_DENSITY, C_HIGH_DENSITY)

        parts = (s11, s12, s21, s22)

        n_draws = 150
        draw_idx = RNG.choice(s11.size, size=n_draws, replace=False)

        for idx in draw_idx:
            axB.plot([1+offs[0], 2+offs[0]], [s11[idx], s21[idx]], color=colors_B[0], lw=0.5, alpha=0.18, zorder=1)
            axB.plot([1+offs[1], 2+offs[1]], [s12[idx], s22[idx]], color=colors_B[1], lw=0.5, alpha=0.18, zorder=1)
        
        m11, m12, m21, m22 = (float(np.median(s)) for s in parts)
    
        axB.set_xticks([1, 2])
        axB.set_xticklabels([f"Low Pred.", f"High Pred."], fontsize=10)
        axB.set_ylabel("Marginal $\\mathit{v}$", fontsize=12)
        axB.tick_params(labelsize=10)

        # Panel B: 4 y-ticks from 2.1 to 3.6 (step 0.5)
        axB.set_yticks(np.arange(2.1, 3.6 + 1e-9, 0.5))
        axB.set_ylim(2.1, 3.6)
        axB.yaxis.set_major_formatter(mpl.ticker.FormatStrFormatter('%.1f'))

        # Panel label (will be aligned after tight_layout)
        _panelB = axB.text(0.0, 1.0, "b", transform=axB.transAxes, ha="left", va="center", fontweight="bold", fontsize=_PANEL_LETTER_FS)

        # Inline labels (match trace colors)
        axB.text(1.2, 2.4, "Low Density", ha="center", va="center", fontsize=10, color=colors_B[0], clip_on=False)
        axB.text(1.4, 3.4, "High Density", ha="center", va="center", fontsize=10, color=colors_B[1], clip_on=False)


        # ==========================================
        # Panel C: Posterior of Δ (Bayesian contrast)
        # ==========================================
        for arr, col, lab in [(delta_long, C_LOW_DENSITY, "Low Density"),
                              (delta_short, C_HIGH_DENSITY, "High Density")]:
            xs, ys = _kde(arr, n=600)
            axC.fill_between(xs, 0, ys, color=col, alpha=0.2)
            lo, hi = _hdi(arr, 0.95)
            mask = (xs >= lo) & (xs <= hi)
            axC.fill_between(xs, 0, ys, where=mask, color=col, alpha=0.5)
            axC.plot(xs, ys, color=col, lw=1.5, alpha=0.8)

        axC.axvline(0, color="black", ls="--", lw=1.2, zorder=1)
        axC.set_xlabel("$\\Delta \\mathit{v}$ (High - Low Pred.)", fontsize=12)
        axC.set_ylabel(
            r"Posterior $p(\Delta \mathit{v})$",
            fontsize=12,
        )
        axC.tick_params(labelsize=10)
        axC.set_ylim(0.0, 1.5)
        axC.set_yticks(np.arange(0.0, 1.5 + 1e-9, 0.5))
        _xc_lo = min(float(np.nanmin(delta_long)), float(np.nanmin(delta_short)))
        _xc_hi = max(float(np.nanmax(delta_long)), float(np.nanmax(delta_short)))
        _xpad_c = 0.12
        axC.set_xlim(min(-1.0 - _xpad_c, _xc_lo - _xpad_c), max(2.0 + _xpad_c, _xc_hi + _xpad_c))
        axC.set_xticks([-1, 0, 1, 2])

        _panelC = axC.text(0.0, 1.0, "c", transform=axC.transAxes, ha="left", va="center", fontweight="bold", fontsize=_PANEL_LETTER_FS)

        axC.text(1.35, 1.2, "Low Density", ha="center", va="center", fontsize=10, color=C_LOW_DENSITY, clip_on=False)
        axC.text(-0.75, 1.2, "High Density", ha="center", va="center", fontsize=10, color=C_HIGH_DENSITY, clip_on=False)

        for ax in (axA, axB, axC):
            _style_spines_offset(ax)

        fig.canvas.draw()
        ren = fig.canvas.get_renderer()

        def _align_panel_x(ax, txt):
            lab = ax.yaxis.label
            bb = lab.get_window_extent(renderer=ren)
            x0, _ = ax.transAxes.inverted().transform((bb.x0, bb.y0))
            txt.set_position((x0, 1.0))

        _align_panel_x(axA, _panelA)
        _align_panel_x(axB, _panelB)
        _align_panel_x(axC, _panelC)
        for _ax in (axA, axB, axC):
            _clip_spines_to_tick_range(_ax)

        fig.canvas.draw()


    return fig


def _plot_combined_a_panel(idata, ax):
    """Combined figure row 2 panel D: main effects of decision boundary a."""
    ta = _traces_8(idata, "a")
    if len(ta) < 8:
        raise RuntimeError("Missing some a(condition) nodes. Expected 8 design cells.")

    with _fig235_rc_context(font_pt_bump=1):
        a_flex = _main_marginal(ta, factor_idx=2, level=TRANS_FLEX)
        a_stab = _main_marginal(ta, factor_idx=2, level=TRANS_STAB)
        a_high_density = _main_marginal(ta, factor_idx=1, level=PROB_HIGH_PERT)
        a_low_density = _main_marginal(ta, factor_idx=1, level=PROB_LOW_PERT)

        arrays = [a_flex, a_stab, a_high_density, a_low_density]
        labels = ["Flexibility", "Stability", "High Density", "Low Density"]
        colors = [C_FLEX, C_STAB, C_HIGH_DENSITY, C_LOW_DENSITY]
        fig = ax.figure

        for i, (arr, col) in enumerate(zip(arrays, colors)):
            vp = ax.violinplot([arr], positions=[i], widths=0.78, showextrema=False, vert=True)
            for b in vp["bodies"]:
                b.set_facecolor(col)
                b.set_alpha(0.5)
                verts = b.get_paths()[0].vertices
                verts[:, 0] = np.maximum(verts[:, 0], float(i))

            n_points = min(2000, arr.size)
            idx = RNG.choice(arr.size, size=n_points, replace=False)
            jitter = RNG.uniform(low=-0.2, high=-0.05, size=n_points)
            ax.scatter(i + jitter, arr[idx], s=4.2, alpha=0.28, color=col, edgecolor="none", zorder=1)

            med = float(np.median(arr))
            lo, hi = _hdi(arr, 0.95)
            ax.plot([i, i], [lo, hi], color="black", lw=1.5, zorder=2)
            ax.scatter(i, med, color="white", edgecolor="black", s=32, zorder=3)

        ax.set_ylabel("Decision boundary ($\\mathit{a}$)", fontsize=12)
        ax.set_yticks(np.arange(1.4, 5.4 + 1e-9, 1.0))
        ax.set_ylim(1.35, 5.45)
        ax.yaxis.set_major_formatter(mpl.ticker.FormatStrFormatter("%.1f"))
        ax.set_xticks([0, 1, 2, 3])
        ax.set_xticklabels(labels, fontsize=10, rotation=15, ha="right")
        ax.set_xlim(-0.55, 3.55)
        ax.tick_params(labelsize=10, axis="both", pad=6)
        ax.axvline(1.5, color="black", lw=1.0, alpha=0.8)
        _style_spines_offset(ax)

        tD = ax.text(0.0, 1.0, "d", transform=ax.transAxes, ha="left", va="center", fontweight="bold", fontsize=_PANEL_LETTER_FS)

        fig.canvas.draw()
        ren = fig.canvas.get_renderer()
        bb = ax.yaxis.label.get_window_extent(renderer=ren)
        x0, _ = ax.transAxes.inverted().transform((bb.x0, bb.y0))
        tD.set_position((x0 + _COMBINED_PANEL_LETTER_X_SHIFT, 1.0))
        _clip_spines_to_tick_range(ax)
        fig.canvas.draw()
        plt.setp(ax.get_xticklabels(), rotation=15, ha="right", rotation_mode="anchor")
    return fig

_COMBINED_PANEL_LETTER_X_SHIFT = -0.035
_PANEL_LETTER_FS = 16



def _align_ddm_panel_letter_to_ylabel(ax, txt, *, x_shift=0.0):
    fig = ax.figure
    fig.canvas.draw()
    ren = fig.canvas.get_renderer()
    bb = ax.yaxis.label.get_window_extent(renderer=ren)
    x0, _ = ax.transAxes.inverted().transform((bb.x0, bb.y0))
    txt.set_position((x0 + x_shift, 1.0))


def _find_panel_letter(ax, letter):
    for _t in ax.texts:
        if _t.get_text() == letter:
            return _t
    return None


def _align_combined_panel_E(fig, axB, axD, panel_e):
    """E: left with B, bottom with D (figure coordinates)."""
    fig.canvas.draw()
    ren = fig.canvas.get_renderer()
    t_b = _find_panel_letter(axB, "b")
    t_d = _find_panel_letter(axD, "d")
    if t_b is None or t_d is None:
        return
    bb_b = t_b.get_window_extent(renderer=ren)
    bb_d = t_d.get_window_extent(renderer=ren)
    inv = fig.transFigure.inverted()
    x_fig, _ = inv.transform((bb_b.x0, bb_b.y0))
    _, y_fig = inv.transform((bb_d.x0, bb_d.y0))
    panel_e.remove()
    fig.text(
        x_fig,
        y_fig,
        "e",
        transform=fig.transFigure,
        ha="left",
        va="bottom",
        fontweight="bold",
        fontsize=_PANEL_LETTER_FS,
        clip_on=False,
    )


def _nudge_combined_ddm_axes(ax_lo, ax_hi, *, dy_down=0.018, shift_left=0.012):
    """Shift E sub-panels; common axes bottom; left half-panel nudged left."""
    p_lo, p_hi = ax_lo.get_position(), ax_hi.get_position()
    y0 = min(p_lo.y0, p_hi.y0) - dy_down
    ax_lo.set_position([p_lo.x0 - shift_left, y0, p_lo.width, p_lo.height])
    ax_hi.set_position([p_hi.x0, y0, p_hi.width, p_hi.height])


def _align_ddm_frame_bottoms(ax_lo, ax_hi):
    """Align rounded DDM frame (FancyBbox) bottom edges in display coordinates."""
    fig = ax_lo.figure
    fig.canvas.draw()

    def _frame_bottom_display(ax):
        ylo, _ = ax.get_ylim()
        y_frame = ylo + 0.03
        return ax.transData.transform((0.0, y_frame))[1]

    b_lo, b_hi = _frame_bottom_display(ax_lo), _frame_bottom_display(ax_hi)
    target = min(b_lo, b_hi)
    inv = fig.transFigure.inverted()
    for ax, b_cur in ((ax_lo, b_lo), (ax_hi, b_hi)):
        dy = target - b_cur
        if abs(dy) < 1.5:
            continue
        p = ax.get_position()
        bb = ax.get_window_extent()
        y0_fig_old = inv.transform((bb.x0, bb.y0))[1]
        y0_fig_new = inv.transform((bb.x0, bb.y0 + dy))[1]
        ax.set_position([p.x0, p.y0 + (y0_fig_new - y0_fig_old), p.width, p.height])


def _align_ddm_time_labels_bottom(ax_lo, ax_hi):
    """Bottom-align the two DDM 'Time' x-axis labels in axes coordinates."""
    fig = ax_lo.figure
    fig.canvas.draw()
    ren = fig.canvas.get_renderer()
    y_axes = []
    for ax in (ax_lo, ax_hi):
        bb = ax.xaxis.label.get_window_extent(renderer=ren)
        _, y = ax.transAxes.inverted().transform((bb.x0 + 0.5 * bb.width, bb.y0))
        y_axes.append(y)
    y_bot = min(y_axes) + 0.026
    for ax in (ax_lo, ax_hi):
        ax.xaxis.set_label_coords(0.5, y_bot)


def _ddm_standalone_panel_h_over_w():
    """Physical height/width of one standalone DDM half-panel."""
    try:
        return float(_DDM_STANDALONE_PANEL_H_OVER_W)
    except NameError:
        plot_w = 7.2 * (0.96 - 0.07)
        plot_h = 3.55 * (0.86 - 0.16)
        return plot_h / (plot_w / (2.0 + 0.38))


def _combined_figure_geometry(
    panel_h_in=2.45,
    *,
    left=0.05,
    right=0.99,
    bottom=0.09,
    top=0.96,
    wspace=0.30,
    hspace=0.38,
):
    """Fig size + gridspec so E keeps standalone w/h; all panels share height panel_h_in.

    Column B/C width follows E (not vice versa); A/D use the same width so A–D share one aspect.
    """
    h_over_w = _ddm_standalone_panel_h_over_w()
    w_e = panel_h_in / h_over_w
    w_a = w_e
    mean_w = (w_a + 2.0 * w_e) / 3.0
    usable_w = w_a + 2.0 * w_e + 2.0 * wspace * mean_w
    usable_h = 2.0 * panel_h_in + hspace * panel_h_in
    fig_w = usable_w / (right - left)
    fig_h = usable_h / (top - bottom)
    gs_kw = dict(
        width_ratios=[w_a, w_e, w_e],
        height_ratios=[1.0, 1.0],
        wspace=wspace,
        hspace=hspace,
        left=left,
        right=right,
        bottom=bottom,
        top=top,
    )
    return (fig_w, fig_h), gs_kw


def plot_combined_figure_v_a_ddm(idata, savepath=None, dpi=600):
    """2×3 layout: row 0 drift-rate panels (A–C); row 1 boundary panel (D) + DDM schematic (E)."""
    with _fig235_rc_context(font_pt_bump=1):
        figsize, gs_kw = _combined_figure_geometry(panel_h_in=2.45)
        fig = plt.figure(figsize=figsize, facecolor="none")
        gs = fig.add_gridspec(2, 3, **gs_kw)
        axA = fig.add_subplot(gs[0, 0])
        axB = fig.add_subplot(gs[0, 1])
        axC = fig.add_subplot(gs[0, 2])
        axD = fig.add_subplot(gs[1, 0])
        axE_lo = fig.add_subplot(gs[1, 1])
        axE_hi = fig.add_subplot(gs[1, 2])

        _plot_combined_v_panels(idata, (axA, axB, axC))
        _plot_combined_a_panel(idata, axD)
        _nudge_combined_ddm_axes(axE_lo, axE_hi)
        draw_ddm_ballistic_on_axes(
            axE_lo,
            axE_hi,
            parent_fig=fig,
            mid_label_n_em=3.4,
            bound_label_x_pad_top=0.02,
            bound_label_x_pad_bot=0.05,
            time_label_y=-0.035,
        )
        _align_ddm_frame_bottoms(axE_lo, axE_hi)
        _align_ddm_time_labels_bottom(axE_lo, axE_hi)

        _panelE = axE_lo.text(
            0.0, 1.0, "e",
            transform=axE_lo.transAxes,
            ha="left", va="center", fontweight="bold", fontsize=15,
        )

        fig.canvas.draw()
        for _ax in (axA, axB, axC, axD):
            for _child in _ax.texts:
                if _child.get_text() in ("a", "b", "c", "d"):
                    _align_ddm_panel_letter_to_ylabel(_ax, _child, x_shift=_COMBINED_PANEL_LETTER_X_SHIFT)
                    break
        _align_combined_panel_E(fig, axB, axD, _panelE)

        if savepath:
            _stem = Path(savepath).with_suffix("")
            _kw = dict(
                dpi=dpi,
                bbox_inches="tight",
                transparent=True,
                facecolor="none",
                edgecolor="none",
            )
            save_figure(fig, str(_stem) + ".png", **_kw)
        return fig


HDDM: pytorch module seems missing. No LAN functionality can be loaded.
It seems that you do not have pytorch installed. You cannot use the network_inspector module.
It seems that you do not have pytorch installed.The HDDMnn, HDDMnnRegressor, HDDMnnStimCoding, HDDMnnRL and HDDMnnRLRegressorclasses will not work
It seems that you do not have pytorch installed.The HDDMnn, HDDMnnRegressor, HDDMnnStimCoding, HDDMnnRL and HDDMnnRLRegressorclasses will not work
It seems that you do not have pytorch installed.The HDDMnn, HDDMnnRegressor and HDDMnnStimCodingclasses will not work
It seems that you do not have pytorch installed.The HDDMnn, HDDMnnRL, HDDMnnRegressor, HDDMnnStimCoding, HDDMnnRL and HDDMnnRLRegressorclasses will not work
It seems that you do not have pytorch installed.The HDDMnn, HDDMnnRegressor, HDDMnnStimCoding, HDDMnnRL and HDDMnnRLRegressorclasses will not work


In [3]:
%matplotlib inline

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch, Rectangle
from matplotlib.transforms import blended_transform_factory
from pathlib import Path

C_LOW_DENSITY = "#85A897"
C_HIGH_DENSITY = "#9B8AA6"
# Drift arrows: colorblind-safe (no green/blue/orange/purple; Paul Tol / Nature style)
C_V_HIGH_PRED = "#C75146"  # brick red
C_V_LOW_PRED = "#B8960C"   # ochre / dark gold
C_V_BALLISTIC = "#CC6677"  # rose
C_BOUND = "#2A2A2A"
C_T_FILL = "#ECECEC"
C_Z_DOT = "#1A1A1A"

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 9,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "figure.dpi": 600,
    "savefig.dpi": 600,
    "figure.facecolor": "none",
    "axes.facecolor": "none",
    "savefig.facecolor": "none",
    "savefig.transparent": True,
})


def _drift_path(t_nd, v, y0, y_bound, t_max):
    tau_hit = (y_bound - y0) / v
    tau_end = min(tau_hit, t_max - t_nd)
    tau = np.linspace(0.0, tau_end, 80)
    return t_nd + tau, y0 + v * tau


def _n_em_data(ax, fontsize, n=0.5):
    """Vertical shift of n font em in data coordinates."""
    fig = ax.figure
    h_in = ax.get_position().height * fig.get_figheight()
    ylo, yhi = ax.get_ylim()
    return n * fontsize / 72.0 * (yhi - ylo) / h_in


def _n_px_data(ax, px):
    """Vertical shift of px screen pixels in data coordinates."""
    fig = ax.figure
    h_in = ax.get_position().height * fig.get_figheight()
    ylo, yhi = ax.get_ylim()
    return (px / fig.dpi) * (yhi - ylo) / h_in


def _n_em_fig(fig, fontsize, n=0.5):
    """Vertical shift of n font em in figure coordinates."""
    return n * fontsize / 72.0 / fig.get_figheight()


def _n_em_axes(ax, fontsize, n=0.5):
    """Shift of n font em in axes fraction coordinates."""
    return n * fontsize / 72.0 / ax.get_position().height


def _y_at_axes_frac(ax, frac):
    """Data y at a fixed fraction of the current axes y-span."""
    ylo, yhi = ax.get_ylim()
    return ylo + frac * (yhi - ylo)


def _dy_axes_frac(ax, dfrac):
    """Data dy corresponding to a fraction of the axes y-span."""
    ylo, yhi = ax.get_ylim()
    return dfrac * (yhi - ylo)


# Footer: bracket lines vs label use separate axes-y (lines stay fixed; label moves alone).
_FOOTER_BRACKET_LINE_Y_AXES = 0.096
_FOOTER_BRACKET_LABEL_Y_AXES = 0.158
_FOOTER_BRACKET_TICK_DY_AXES = 0.011
_FOOTER_EVIDENCE_Y_AXES = 0.024

# E-panel body text bump (+1 pt); excludes title, subtitle, and axis labels.
_PANEL_E_ANNOT_FS_BUMP = 1

# Paired DDM panels: identical y-limits and rounded-frame geometry (data coords).
_DDM_REF_A_HALF = 0.525
_DDM_Y_LIM = (-_DDM_REF_A_HALF - 0.55, _DDM_REF_A_HALF + 0.75)
_DDM_FRAME_Y0 = -_DDM_REF_A_HALF - 0.52
_DDM_FRAME_H = 2.0 * _DDM_REF_A_HALF + 1.2


def _set_xlabel_at_axis(ax, text="Time", label_y=-0.06, fontsize=12):
    """Place x-axis label just below the axis line (above footer area)."""
    ax.set_xlabel(text, fontsize=fontsize)
    ax.xaxis.set_label_coords(0.5, label_y)


def _rot_xy(x, y, cx, cy, deg):
    """Rotate point (x, y) about (cx, cy) by deg (CW negative)."""
    th = np.deg2rad(deg)
    dx, dy = x - cx, y - cy
    c, s = np.cos(th), np.sin(th)
    return cx + dx * c - dy * s, cy + dx * s + dy * c


def _draw_context_gating_arc(ax, t_nd, y0, v_hi, v_lo, t_axis_max=3.6, fs_bump=0):
    """Gating arc between drift lines; rotation via endpoint math (no Affine2D)."""
    t1 = t_nd + 0.22 * (t_axis_max - t_nd)
    dt = t1 - t_nd
    y_hi = y0 + v_hi * dt
    y_lo = y0 + v_lo * dt
    y_mid = 0.5 * (y_hi + y_lo)
    cx, cy = t1, y_mid
    _sc = 1.28
    _x0, _y0 = cx + _sc * (t1 - cx), cy + _sc * (y_lo - cy)
    _x1, _y1 = cx + _sc * (t1 - cx), cy + _sc * (y_hi - cy)
    x0, y0a = _rot_xy(_x0, _y0, cx, cy, -130)
    x1, y1a = _rot_xy(_x1, _y1, cx, cy, -130)
    ax.add_patch(
        FancyArrowPatch(
            (x0, y0a),
            (x1, y1a),
            arrowstyle="<|-|>",
            mutation_scale=9,
            connectionstyle="arc3,rad=-0.55",
            color="#1A1A1A",
            lw=0.68,
            zorder=30,
            clip_on=False,
        )
    )
    _fs_gate = 6.8 + fs_bump
    ax.text(
        t1 + 0.14,
        y_mid - _n_em_data(ax, _fs_gate, 2.0),
        "Context-dependent\nGating",
        fontsize=_fs_gate,
        color="#1A1A1A",
        ha="left",
        va="center",
        linespacing=1.15,
        zorder=31,
        clip_on=False,
    )


def _bracket_h(
    ax, x0, x1, y_line_axes, label, color="#404040", lw=0.7, fs=8, label_x_frac=0.5, *, y_label_axes=None,
):
    """Horizontal bracket: x in data; line y fixed in axes fraction; label y independent."""
    trans = blended_transform_factory(ax.transData, ax.transAxes)
    dt = _FOOTER_BRACKET_TICK_DY_AXES
    y_lab = y_label_axes if y_label_axes is not None else y_line_axes + 0.046
    ax.plot([x0, x1], [y_line_axes, y_line_axes], transform=trans, color=color, lw=lw, clip_on=False)
    ax.plot([x0, x0], [y_line_axes - dt, y_line_axes + dt], transform=trans, color=color, lw=lw, clip_on=False)
    ax.plot([x1, x1], [y_line_axes - dt, y_line_axes + dt], transform=trans, color=color, lw=lw, clip_on=False)
    _va = "center" if "\n" in label else "bottom"
    ax.text(
        x0 + label_x_frac * (x1 - x0),
        y_lab,
        label,
        transform=trans,
        ha="center",
        va=_va,
        fontsize=fs,
        color=color,
        linespacing=1.15,
        clip_on=False,
    )


def _bracket_v(
    ax, x, y0, y1, label, color="#404040", lw=0.7, fs=8, *, text_ha="left", text_va="center", linespacing=1.15,
):
    ax.plot([x, x], [y0, y1], color=color, lw=lw, clip_on=False)
    ax.plot([x - 0.03, x + 0.03], [y0, y0], color=color, lw=lw, clip_on=False)
    ax.plot([x - 0.03, x + 0.03], [y1, y1], color=color, lw=lw, clip_on=False)
    _tx = x if text_ha == "center" else x + 0.06
    ax.text(
        _tx,
        0.5 * (y0 + y1),
        label,
        ha=text_ha,
        va=text_va,
        fontsize=fs,
        color=color,
        rotation=90,
        linespacing=linespacing,
    )


def draw_ddm_panel(
    ax,
    *,
    title,
    accent,
    a_half,
    t_nd,
    z_frac,
    drifts,
    drift_colors,
    drift_labels,
    subtitle=None,
    show_legend=True,
    merged_ballistic=False,
    z_label_x_em=0.0,
    z_label_y_em_up=0.0,
    z_label_legacy_offset=False,
    context_gating_arc=False,
    bound_label_x_pad_top=0.08,
    bound_label_x_pad_bot=0.08,
    bound_label_x_nudge_bot=0.0,
    boundary_line_expand=0.0,
    clamp_nd_fill_to_bounds=False,
    show_ylabel=True,
    ylabel_labelpad=4,
    time_label_y=-0.06,
    axis_label_fs=12,
    paired_layout=False,
    annot_fs_bump=0,
):
    t_axis_max = 3.6
    _b = annot_fs_bump
    y0 = (2 * z_frac - 1) * a_half
    y_top, y_bot = a_half, -a_half
    _yb_top = y_top + boundary_line_expand
    _yb_bot = y_bot - boundary_line_expand

    ax.set_xlim(-0.15, t_axis_max + 0.35)
    if paired_layout:
        ax.set_ylim(*_DDM_Y_LIM)
    else:
        ax.set_ylim(y_bot - 0.55, y_top + 0.75)
    ax.axhline(0, color="#BBBBBB", lw=0.5, zorder=0)

    if clamp_nd_fill_to_bounds:
        if boundary_line_expand > 0:
            _t_y0, _t_h = _yb_bot, _yb_top - _yb_bot
        else:
            _t_y0, _t_h = y_bot, y_top - y_bot
    else:
        _t_y0, _t_h = y_bot - 0.08, (y_top - y_bot) + 0.16
    ax.add_patch(
        Rectangle(
            (0, _t_y0),
            t_nd,
            _t_h,
            facecolor=C_T_FILL,
            edgecolor="none",
            zorder=0,
        )
    )
    _frame_x0 = -0.12
    _frame_x1 = _frame_x0 + t_axis_max + 0.5
    if paired_layout:
        _frame_y0 = _DDM_FRAME_Y0
        _frame_h = _DDM_FRAME_H
    else:
        _frame_y0 = y_bot - 0.52
        _frame_h = (y_top - y_bot) + 1.2
    ax.plot(
        [t_nd, t_nd],
        [_frame_y0, _frame_y0 + _frame_h],
        color="#888888",
        ls=(0, (4, 3)),
        lw=0.7,
        zorder=1,
        clip_on=True,
    )

    for yb in (_yb_top, _yb_bot):
        ax.plot(
            [_frame_x0, _frame_x1],
            [yb, yb],
            color=C_BOUND,
            lw=1.6,
            zorder=2,
            clip_on=False,
            solid_capstyle="butt",
        )
    _fs_bound = 7.5 + _b
    _dy = _n_em_data(ax, _fs_bound, 0.5)
    _dy_dn = _n_em_data(ax, _fs_bound, 0.35)
    _dy_up = _n_em_data(ax, _fs_bound, 0.35)
    ax.text(
        t_axis_max + bound_label_x_pad_top,
        _yb_top + _dy + _dy_up - _n_em_data(ax, _fs_bound, 0.3),
        r"$+\mathit{a}/2$",
        va="center",
        ha="left",
        fontsize=_fs_bound,
        color=C_BOUND,
    )
    ax.text(
        t_axis_max + bound_label_x_pad_bot + bound_label_x_nudge_bot,
        _yb_bot - _dy - _dy_dn,
        r"$-\mathit{a}/2$",
        va="center",
        ha="left",
        fontsize=_fs_bound,
        color=C_BOUND,
    )

    ax.plot(
        t_nd,
        y0,
        "o",
        ms=7,
        color=C_Z_DOT,
        zorder=6,
        markeredgecolor="white",
        markeredgewidth=0.6,
    )
    _fs_z = 7.5 + _b
    _z_x_pt = 12 + z_label_x_em * _fs_z
    if z_label_legacy_offset:
        # Left panel: fixed offset-points layout (do not change with right panel).
        ax.annotate(
            r"$z = 0.5$" + "\n(start)",
            xy=(t_nd, y0),
            xytext=(_z_x_pt, -10),
            textcoords="offset points",
            fontsize=_fs_z,
            color=C_Z_DOT,
            arrowprops=dict(arrowstyle="->", color=C_Z_DOT, lw=0.7),
            ha="center",
            va="top",
        )
    else:
        # Right panel: data-y shift; positive z_label_y_em_up = up, negative = down.
        _z_y = y0 + _n_em_data(ax, _fs_z, 10.0 / _fs_z + z_label_y_em_up)
        ax.annotate(
            r"$z = 0.5$" + "\n(start)",
            xy=(t_nd, y0),
            xytext=(_z_x_pt, _z_y),
            textcoords=("offset points", "data"),
            fontsize=_fs_z,
            color=C_Z_DOT,
            arrowprops=dict(arrowstyle="->", color=C_Z_DOT, lw=0.7),
            ha="center",
            va="bottom",
        )

    _gating_lab_y = None
    if context_gating_arc and len(drifts) >= 2:
        # Shared anchor: label bottom edge ~5 px above the upper boundary line.
        _gating_lab_y = _yb_top + _n_px_data(ax, 5)

    for k, (v, col, lab) in enumerate(zip(drifts, drift_colors, drift_labels)):
        x, y = _drift_path(t_nd, v, y0, _yb_top, t_axis_max)
        lw = 2.2 if merged_ballistic else 1.9
        ax.plot(x, y, color=col, lw=lw, solid_capstyle="round", zorder=4)
        i = int(0.45 * len(x))
        if i < len(x) - 2:
            ax.annotate(
                "",
                xy=(x[i + 1], y[i + 1]),
                xytext=(x[i], y[i]),
                arrowprops=dict(arrowstyle="->", color=col, lw=1.4),
                zorder=5,
            )
        _lab_dx, _lab_dy = 0.05, 0.0
        _fs_lab = 7 + _b
        _ha = "left"
        _va = "center"
        if context_gating_arc and len(drifts) >= 2:
            _em = lambda n: _n_em_data(ax, _fs_lab, n)
            _ha = "center"
            _va = "bottom"
            if k == 0:
                _lab_dx -= 0.26
                _lab_dx -= _em(1.0)
                _lab_dx += _em(1.0)
                _lab_dx += _em(1.0)
                _lab_dx += _em(1.0)
                _lab_dy = _gating_lab_y - y[-1]
            elif k == 1:
                _lab_dx -= 0.13
                _lab_dx += _em(2.0)
                _lab_dx += _em(1.35)
                _lab_dy = _gating_lab_y - y[-1]
        elif merged_ballistic:
            _d = _n_em_data(ax, _fs_lab, 1.0)
            _lab_dx -= _d
            _lab_dy += _d
        _fs_out = (8 + _b) if merged_ballistic else _fs_lab
        ax.text(
            x[-1] + _lab_dx,
            y[-1] + _lab_dy,
            lab,
            fontsize=_fs_out,
            color=col,
            ha=_ha,
            va=_va,
            fontweight="normal",
            linespacing=1.15,
        )

    _bracket_h(
        ax, 0, t_nd, _FOOTER_BRACKET_LINE_Y_AXES, "non-decision\ntime ($t$)",
        fs=7.5 + _b, label_x_frac=0.66, y_label_axes=_FOOTER_BRACKET_LABEL_Y_AXES,
    )
    _bracket_v(
        ax, t_axis_max - 0.25, _yb_bot, _yb_top,
        "compressed\nboundary ($a$)" if merged_ballistic else r"boundary ($a$)",
        fs=7.5 + _b,
        text_ha="center" if merged_ballistic else "left",
        text_va="center",
    )
    _ev_trans = blended_transform_factory(ax.transData, ax.transAxes)
    ax.text(
        t_nd + 0.55 * (t_axis_max - t_nd),
        _FOOTER_EVIDENCE_Y_AXES,
        "evidence accumulation",
        transform=_ev_trans,
        ha="center",
        va="bottom",
        fontsize=7 + _b,
        color="#666666",
        style="italic",
        clip_on=False,
    )

    _set_xlabel_at_axis(ax, "Time", label_y=time_label_y, fontsize=axis_label_fs)
    if show_ylabel:
        ax.set_ylabel("Accumulated Evidence", labelpad=ylabel_labelpad, fontsize=axis_label_fs)
    else:
        ax.set_ylabel("")
        ax.yaxis.label.set_visible(False)
    ax.set_title(title, fontweight="bold", color=accent, pad=10.5, linespacing=1.2)
    if subtitle:
        ax.text(
            0.5,
            1.04,
            subtitle,
            transform=ax.transAxes,
            ha="center",
            va="top",
            fontsize=8,
            color="#555555",
            style="italic",
            linespacing=1.55,
        )

    for side in ("top", "right", "bottom", "left"):
        ax.spines[side].set_visible(False)
    ax.set_xticks([0, t_nd, t_axis_max])
    ax.set_xticklabels(["0", r"$t$", ""])
    ax.set_yticks([])
    ax.tick_params(
        axis="x",
        bottom=True,
        labelbottom=True,
        top=False,
        labeltop=False,
        length=3.5,
        width=0.7,
        colors="#404040",
    )
    ax.tick_params(axis="y", left=False, labelleft=False)

    ax.add_patch(
        FancyBboxPatch(
            (_frame_x0, _frame_y0),
            t_axis_max + 0.5,
            _frame_h,
            boxstyle="round,pad=0.02,rounding_size=0.08",
            linewidth=1.2,
            edgecolor=accent,
            facecolor="none",
            zorder=0,
            clip_on=False,
        )
    )

    if context_gating_arc and len(drifts) >= 2:
        _draw_context_gating_arc(ax, t_nd, y0, drifts[0], drifts[1], t_axis_max=t_axis_max, fs_bump=_b)



def _equalize_ddm_panel_axes(ax_lo, ax_hi):
    """Match E half-panel axes boxes (width, height, bottom) in figure coordinates."""
    p_lo, p_hi = ax_lo.get_position(), ax_hi.get_position()
    w = max(p_lo.width, p_hi.width)
    h = max(p_lo.height, p_hi.height)
    y0 = min(p_lo.y0, p_hi.y0)
    ax_lo.set_position([p_lo.x0, y0, w, h])
    ax_hi.set_position([p_hi.x0, y0, w, h])


def draw_ddm_ballistic_on_axes(
    ax_lo,
    ax_hi,
    parent_fig=None,
    *,
    mid_label_n_em=2.2,
    mid_x_center=0.5,
    mid_panel_shift_left=0.014,
    bound_label_x_pad_top=0.08,
    bound_label_x_pad_bot=0.08,
    bound_label_x_nudge_bot=-0.04,
    boundary_line_expand_hi=0.102,
    bound_label_x_pad=None,
    time_label_y=-0.06,
    axis_label_fs=12,
    annot_fs_bump=_PANEL_E_ANNOT_FS_BUMP,
):
    """Draw two-panel DDM ballistic schematic on supplied axes (composite layout)."""
    if bound_label_x_pad is not None:
        bound_label_x_pad_top = bound_label_x_pad_bot = bound_label_x_pad
    Z, T_ND = 0.5, 0.55
    A_LOW, A_HIGH = 1.05, 0.52
    V_HI, V_LO, V_BAL = 0.72, 0.38, 0.88

    draw_ddm_panel(
        ax_lo,
        title="Low Perturbation Density",
        subtitle="Context-dependent Gating of Drift(v)",
        accent=C_LOW_DENSITY,
        a_half=A_LOW / 2,
        t_nd=T_ND,
        z_frac=Z,
        drifts=[V_HI, V_LO],
        drift_colors=[C_V_HIGH_PRED, C_V_LOW_PRED],
        drift_labels=[
            r"High Pred." + "\n" + r"(Gated $v_{\mathrm{high}}$)",
            r"Low Pred." + "\n" + r"(Gated $v_{\mathrm{low}}$)",
        ],
        show_legend=False,
        context_gating_arc=True,
        z_label_x_em=1.0,
        z_label_legacy_offset=True,
        bound_label_x_pad_top=bound_label_x_pad_top,
        bound_label_x_pad_bot=bound_label_x_pad_bot,
        bound_label_x_nudge_bot=bound_label_x_nudge_bot,
        ylabel_labelpad=10.5,
        time_label_y=time_label_y,
        axis_label_fs=axis_label_fs,
        paired_layout=True,
        clamp_nd_fill_to_bounds=True,
        annot_fs_bump=annot_fs_bump,
    )
    draw_ddm_panel(
        ax_hi,
        title="High Perturbation Density",
        subtitle="Contextual Decoupling / Ballistic Mode",
        accent=C_HIGH_DENSITY,
        a_half=A_HIGH / 2,
        t_nd=T_ND,
        z_frac=Z,
        drifts=[V_BAL],
        drift_colors=[C_V_BALLISTIC],
        drift_labels=[r"$\mathit{v}$ (context-blind)"],
        show_legend=False,
        merged_ballistic=True,
        z_label_x_em=1.5,
        z_label_y_em_up=-4.0,
        bound_label_x_pad_top=bound_label_x_pad_top,
        bound_label_x_pad_bot=bound_label_x_pad_bot,
        bound_label_x_nudge_bot=bound_label_x_nudge_bot,
        boundary_line_expand=boundary_line_expand_hi,
        clamp_nd_fill_to_bounds=True,
        show_ylabel=False,
        time_label_y=time_label_y,
        axis_label_fs=axis_label_fs,
        paired_layout=True,
        annot_fs_bump=annot_fs_bump,
    )
    _equalize_ddm_panel_axes(ax_lo, ax_hi)
    if mid_panel_shift_left:
        _p_hi = ax_hi.get_position()
        ax_hi.set_position([_p_hi.x0 - mid_panel_shift_left, _p_hi.y0, _p_hi.width, _p_hi.height])
    _y_bound_hi = A_HIGH / 2 + boundary_line_expand_hi
    for v, col in ((V_HI, C_V_HIGH_PRED), (V_LO, C_V_LOW_PRED)):
        x, y = _drift_path(T_ND, v, 0, _y_bound_hi, 3.6)
        ax_hi.plot(x, y, color=col, lw=1.0, ls=(0, (5, 4)), alpha=0.35, zorder=3)
    ax_hi.text(
        1.42,
        0.14,
        r"$\Delta \mathit{v} \approx 0$",
        fontsize=7 + annot_fs_bump,
        color="#777777",
        ha="center",
        style="italic",
    )

    fig = parent_fig or ax_lo.figure
    p_lo, p_hi = ax_lo.get_position(), ax_hi.get_position()
    # Center arrow + label in the gap between L/R panel borders (figure coords).
    x_gap_lo, x_gap_hi = p_lo.x1, p_hi.x0
    y_mid = 0.5 * (p_lo.y0 + p_lo.y1)
    _x_mid = 0.5 * (x_gap_lo + x_gap_hi)
    _gap_span = max(x_gap_hi - x_gap_lo, 1e-6)
    _arrow_half = 0.22 * _gap_span
    _x_a0 = _x_mid - _arrow_half
    _x_a1 = _x_mid + _arrow_half
    fig.patches.append(
        FancyArrowPatch(
            (_x_a0, y_mid),
            (_x_a1, y_mid),
            transform=fig.transFigure,
            arrowstyle="-|>",
            mutation_scale=12,
            lw=1.2,
            color="#555555",
            clip_on=False,
        )
    )
    _fs_mid = 7.5 + annot_fs_bump
    fig.text(
        _x_mid,
        y_mid - _n_em_fig(fig, _fs_mid, mid_label_n_em),
        r"$a \downarrow$" + "\n" + r"$v \times$ context" + "\n" + "abolished",
        ha="center",
        fontsize=_fs_mid,
        color="#444444",
        linespacing=1.2,
    )


# DDM half-panel aspect ratio for combined-figure geometry.
_DDM_STANDALONE_PANEL_H_OVER_W = (3.55 * (0.86 - 0.16)) / ((7.2 * (0.96 - 0.07)) / (2.0 + 0.38))


In [4]:
# ==============================
# Run: generate PPC figure + combined panel
# ==============================

hddm_df_use = resolve_hddm_data()
idata_use = resolve_best_idata()

print(f"Behavioral data: {hddm_df_use.shape[0]} trials")
print(f"Posterior: {WINNER_MODEL} ({list(idata_use.posterior.data_vars)[:3]} …)")

fig_ppc = plot_figure_ppc(
    idata_use,
    hddm_df_use,
    savepath=os.path.join(FIG_OUTDIR, "Figure_PPC_DefectiveDensity_2x4.png"),
)

fig_combined = plot_combined_figure_v_a_ddm(
    idata_use,
    savepath=os.path.join(FIG_OUTDIR, "Figure_combined_v_a_DDM_2x3"),
)

print("Saved figures to:", os.path.abspath(FIG_OUTDIR))
# Avoid blocking under nbconvert / pipeline (MPLBACKEND=Agg or env flag).
_noninteractive = (
    os.environ.get("HDDM_PIPELINE_NONINTERACTIVE", "").strip() == "1"
    or str(mpl.get_backend()).lower() == "agg"
)
if not _noninteractive:
    plt.show()
else:
    plt.close("all")


Loading InferenceData: C:\Users\lei\AppData\Local\Temp\hddm_analysis_idata\hddm_va_vat_combined.nc


Behavioral data: 5160 trials
Posterior: hddm_va_vat (['v(Long.prob_same.flexibility)', 'v(Long.prob_same.stability)', 'v(Long.prob_diff.flexibility)'] …)


C:\Users\lei\AppData\Local\Temp\ipykernel_13208\1083583985.py:459: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(rect=(0, 0.08, 1, 0.95))


Saved: D:\文件同步\文件\数据分析结果\data_code_figure\hddm_analysis\figures\Figure_PPC_DefectiveDensity_2x4.png


Saved: D:\文件同步\文件\数据分析结果\data_code_figure\hddm_analysis\figures\Figure_PPC_DefectiveDensity_2x4.pdf


...

Saved: D:\文件同步\文件\数据分析结果\data_code_figure\hddm_analysis\figures\Figure_combined_v_a_DDM_2x3.png


Saved: D:\文件同步\文件\数据分析结果\data_code_figure\hddm_analysis\figures\Figure_combined_v_a_DDM_2x3.pdf


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Saved: D:\文件同步\文件\数据分析结果\data_code_figure\hddm_analysis\figures\Figure_combined_v_a_DDM_2x3.eps


Saved: D:\文件同步\文件\数据分析结果\data_code_figure\hddm_analysis\figures\Figure_combined_v_a_DDM_2x3.svg
Saved figures to: D:\文件同步\文件\数据分析结果\data_code_figure\hddm_analysis\figures
